In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob

from limb_fitting import *

In [6]:
files = sorted(glob.glob('/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-09-23/*.fits.gz'))

files

['/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-09-23/solo_L1_phi-fdt-ilam_20250923T000503_V202512011636C_0569230100.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-09-23/solo_L1_phi-fdt-ilam_20250923T001105_V202511212307C_0569230125.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-09-23/solo_L1_phi-fdt-ilam_20250923T001705_V202511212307C_0569230150.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-09-23/solo_L1_phi-fdt-ilam_20250923T002305_V202511212307C_0569230175.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-09-23/solo_L1_phi-fdt-ilam_20250923T002905_V202511212307C_0569230200.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-09-23/solo_L1_phi-fdt-ilam_20250923T003505_V202511212307C_0569230225.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-09-23/solo_L1_phi-fdt-ilam_20250923T004105_V202511212338C_0569230250.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025

In [92]:
file = files[-1]

with fits.open(file) as hdul:
    image  = hdul[0].data[0]
    header = hdul[0].header

In [93]:
edges = find_edges(image)
x, y = np.where(edges)
x, y = filter_outliers(x, y, acc=2)

In [94]:
plt.figure(figsize=(10,10))
plt.imshow(~edges, 'gray')
plt.plot(y, x, '.', ms=0.5, color='red')

In [95]:
plt.figure(figsize=(10,10))
plt.imshow(image)

In [121]:
class Ellipse(list):
    def __init__(self, *items):
        super().__init__(items)

    @classmethod
    def from_matrix(cls, A):
        coeff = A[0, 0], A[0, 1] + A[1, 0], A[1, 1], A[0, 2] + A[2, 0], A[1, 2] + A[2, 1], A[2, 2]
        return cls(*coeff)

    @property
    def matrix(self):
        A, B, C, D, E, F = self
        return np.array([[A, B / 2, D / 2],
                         [B / 2, C, E / 2],
                         [D / 2, E / 2, F]])

    @property
    def center(self):
        A, B, C, D, E, F = self
        Q = B ** 2 - 4 * A * C
        return (2 * C * D - B * E) / Q, (2 * A * E - B * D) / Q

    @property
    def major(self, minor=False):
        A, B, C, D, E, F = self

        Q = B ** 2 - 4 * A * C
        P = np.sqrt((A - C) ** 2 + B ** 2)
        R = 2 * (A * E ** 2 + C * D ** 2 - B * D * E + Q * F)
        return np.max([- np.sqrt(R * (A + C + P)) / Q, - np.sqrt(R * (A + C - P)) / Q], axis=0)

    @property
    def minor(self):
        A, B, C, D, E, F = self

        Q = B ** 2 - 4 * A * C
        P = np.sqrt((A - C) ** 2 + B ** 2)
        R = 2 * (A * E ** 2 + C * D ** 2 - B * D * E + Q * F)
        return np.min([- np.sqrt(R * (A + C + P)) / Q, - np.sqrt(R * (A + C - P)) / Q], axis=0)

    @property
    def radius(self):
        return (self.minor + self.major) / 2

    @property
    def theta(self):
        A, B, C, D, E, F = self
        theta = np.arctan2(-B, C - A) / 2
        return theta

    @property
    def eccentricity(self):
        a, b = self.major, self.minor
        return np.sqrt(1 - (b / a) ** 2)

    @property
    def flatness(self):
        a, b = self.major, self.minor
        return 1 - b / a

    def patch(self, **kwargs):
        from matplotlib import patches
        return patches.Ellipse(self.center, 2 * self.major, 2 * self.minor,
                               angle=self.theta * 180 / np.pi, fill=False, **kwargs)


def fit_ellipse(x, y):
    mx, my = np.mean(x), np.mean(y)
    sx, sy = np.std(x), np.std(y)

    M = np.array([[1 / sx, 0, -mx / sx],
                  [0, 1 / sy, -my / sy],
                  [0, 0, 1]])
    x_, y_, _ = M @ np.array([x, y, np.ones_like(x)])

    A = np.array([x_ ** 2, x_ * y_, y_ ** 2, x_, y_])
    B = np.sum(A, axis=1)
    a, b, c, d, e = B @ np.linalg.inv(A @ A.T)

    Q = np.array([[a, b / 2, d / 2],
                  [b / 2, c, e / 2],
                  [d / 2, e / 2, -1]])
    Q = M.T @ Q @ M
    return Ellipse(Q[0, 0], Q[0, 1] + Q[1, 0], Q[1, 1], Q[0, 2] + Q[2, 0], Q[1, 2] + Q[2, 1], Q[2, 2])

In [122]:
ellipse = fit_ellipse(x, y)

In [125]:
fig, ax = plt.subplots(figsize=(10,10))
ax.imshow(~(edges.T), 'gray')
ax.add_patch(ellipse.patch(color='red'))